# 结构化输出

结构化输出允许代理以特定且可预测的格式返回数据。您无需解析自然语言响应，即可获得以 JSON 对象、Pydantic 模型或数据类形式存在的结构化数据，您的应用程序可以直接使用这些数据。

LangChaincreate_agent能够自动处理结构化输出。用户设置所需的结构化输出模式，当模型生成结构化数据时，这些数据会被捕获、验证，并以'structured_response'代理状态键的形式返回。

```shell

def create_agent(
    ...
    response_format: Union[
        ToolStrategy[StructuredResponseT],
        ProviderStrategy[StructuredResponseT],
        type[StructuredResponseT],
    ]
```

## 回复格式
控制代理如何返回结构化数据：

- ToolStrategy[StructuredResponseT]使用工具调用进行结构化输出
- ProviderStrategy[StructuredResponseT]使用提供商原生结构化输出
- type[StructuredResponseT]模式类型 - 根据模型功能自动选择最佳策略
- None无结构化输出

当直接提供模式类型时，LangChain 会自动选择：
- ProviderStrategy对于支持原生结构化输出的模型（例如OpenAI、Grok）
- ToolStrategy适用于所有其他型号

结构化响应以structured_response代理最终状态的键返回。

## 供应商策略
部分模型提供商通过其 API 原生支持结构化输出（目前仅支持 OpenAI 和 Grok）。如果可用，这是最可靠的方法。
要使用此策略，请配置`ProviderStrategy`：

```python
class ProviderStrategy(Generic[SchemaT]):
    schema: type[SchemaT]

```

schema required

定义结构化输出格式的模式。支持：
- Pydantic models: BaseModel带有字段验证的子类
- Dataclasses: 带有类型注解的 Python 数据类
- TypedDict: 类型化字典类
- JSON Schema: 包含 JSON Schema 规范的字典

当您将模式类型直接传递给 create_agent.response_format 时，LangChain 会自动使用 ProviderStrategy，并且模型支持本机结构化输出：

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

tools = []
class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    tools=tools,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

提供者原生结构化输出具有高可靠性和严格的验证机制，因为模型提供者会强制执行模式。如果可用，请使用此功能。

> 如果提供商原生支持您所选模型的结构化输出，则在功能上等效于编写 `<script>`response_format=ProductReview而不是 `<script>` response_format=ToolStrategy(ProductReview)。无论哪种情况，如果不支持结构化输出，代理都会回退到工具调用策略。

## 工具调用策略
对于不支持原生结构化输出的模型，LangChain 使用工具调用来实现相同的结果。这适用于所有支持工具调用的模型，而大多数现代模型都支持工具调用。


要使用此策略，请配置ToolStrategy：


```python

class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    tool_message_content: str | None
    handle_errors: Union[
        bool,
        str,
        type[Exception],
        tuple[type[Exception], ...],
        Callable[[Exception], str],
    ]
```

schema required

定义结构化输出格式的模式。支持：

- Pydantic models: BaseModel带有字段验证的子类
- Dataclasses: 带有类型注解的 Python 数据类
- TypedDict: 类型化字典类
- JSON Schema: 包含 JSON Schema 规范的字典
- Union types: 多种模式选项。模型将根据上下文选择最合适的模式。

### 工具消息内容
用于在生成结构化输出时返回工具消息的自定义内容。如果未提供，则默认显示结构化响应数据的消息。

### 处理错误
结构化输出验证失败的错误处理策略。默认为True.
- True使用默认错误模板捕获所有错误
- str使用此自定义消息捕获所有错误
- type[Exception]仅使用默认消息捕获此异常类型
- tuple[type[Exception], ...]仅捕获这些类型的异常，并使用默认消息
- Callable[[Exception], str]：返回错误消息的自定义函数
- False不重试，让异常传播

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductReview(BaseModel):
    """Analysis of a product review."""
    rating: int | None = Field(description="The rating of the product", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")
    key_points: list[str] = Field(description="The key points of the review. Lowercase, 1-3 words each.")

agent = create_agent(
    model="gpt-5",
    tools=tools,
    response_format=ToolStrategy(ProductReview)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})
result["structured_response"]
# ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])

### 自定义工具消息内容
该tool_message_content参数允许您自定义生成结构化输出时显示在对话历史记录中的消息：

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """Action items extracted from a meeting transcript."""
    task: str = Field(description="The specific task to be completed")
    assignee: str = Field(description="Person responsible for the task")
    priority: Literal["low", "medium", "high"] = Field(description="Priority level")

agent = create_agent(
    model="gpt-5",
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes!"
    )
)

agent.invoke({
    "messages": [{"role": "user", "content": "From our meeting: Sarah needs to update the project timeline as soon as possible"}]
})

================================ Human Message =================================     

From our meeting: Sarah needs to update the project timeline as soon as possible      
================================== Ai Message ==================================     
Tool Calls:     
  MeetingAction (call_1)     
 Call ID: call_1     
  Args:     
    task: Update the project timeline     
    assignee: Sarah     
    priority: high     
================================= Tool Message =================================     
Name: MeetingAction     

Action item captured and added to meeting notes!     

如果没有tool_message_content，我们的最终结果ToolMessage将是： 

================================= Tool Message =================================.    
Name: MeetingAction.    

Returning structured response: {'task': 'update the project timeline', 'assignee': 'Sarah', 'priority': 'high'}     

### 错误处理
模型在通过工具调用生成结构化输出时可能会出错。LangChain 提供智能重试机制来自动处理这些错误。
#### 架构验证错误
当结构化输出与预期模式不符时，代理会提供具体的错误反馈：

In [ ]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str = Field(description="Person's name")
    email: str = Field(description="Email address")

class EventDetails(BaseModel):
    event_name: str = Field(description="Name of the event")
    date: str = Field(description="Event date")

agent = create_agent(
    model="gpt-5",
    tools=[],
    response_format=ToolStrategy(Union[ContactInfo, EventDetails])  # Default: handle_errors=True
)

agent.invoke({
    "messages": [{"role": "user", "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

================================ Human Message =================================     

Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th     
None     
================================== Ai Message ==================================     
Tool Calls:     
  ContactInfo (call_1)     
 Call ID: call_1     
  Args:     
    name: John Doe     
    email: john@email.com     
  EventDetails (call_2)     
 Call ID: call_2     
  Args:     
    event_name: Tech Conference     
    date: March 15th     
================================= Tool Message =================================     
Name: ContactInfo     

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.     
 Please fix your mistakes.     
================================= Tool Message =================================     
Name: EventDetails     

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.     
 Please fix your mistakes.     
================================== Ai Message ==================================     
Tool Calls:     
  ContactInfo (call_3)     
 Call ID: call_3     
  Args:     
    name: John Doe     
    email: john@email.com     
================================= Tool Message =================================     
Name: ContactInfo     

Returning structured response: {'name': 'John Doe', 'email': 'john@email.com'}     

#### 架构验证错误
当结构化输出与预期模式不符时，代理会提供具体的错误反馈：

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ProductRating(BaseModel):
    rating: int | None = Field(description="Rating from 1-5", ge=1, le=5)
    comment: str = Field(description="Review comment")

agent = create_agent(
    model="gpt-5",
    tools=[],
    response_format=ToolStrategy(ProductRating),  # Default: handle_errors=True
    system_prompt="You are a helpful assistant that parses product reviews. Do not make any field or value up."
)

agent.invoke({
    "messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!"}]
})

================================ Human Message =================================      

Parse this: Amazing product, 10/10!     
================================== Ai Message ==================================     
Tool Calls:     
  ProductRating (call_1)     
 Call ID: call_1     
  Args:     
    rating: 10     
    comment: Amazing product     
================================= Tool Message =================================     
Name: ProductRating     

Error: Failed to parse structured output for tool 'ProductRating': 1 validation error for ProductRating.rating     
  Input should be less than or equal to 5 [type=less_than_equal, input_value=10, input_type=int].     
 Please fix your mistakes.     
================================== Ai Message ==================================     
Tool Calls:     
  ProductRating (call_2)     
 Call ID: call_2     
  Args:     
    rating: 5     
    comment: Amazing product     
================================= Tool Message =================================     
Name: ProductRating     

Returning structured response: {'rating': 5, 'comment': 'Amazing product'}     

#### 错误处理策略
您可以使用以下参数自定义错误处理方式handle_errors：
##### 自定义错误信息：

In [ ]:
ToolStrategy(
    schema=ProductRating,
    handle_errors="Please provide a valid rating between 1-5 and include a comment."
)

如果handle_errors是一个字符串，代理将始终提示模型使用固定的工具消息重试：

================================= Tool Message =================================.  
Name: ProductRating. 

Please provide a valid rating between 1-5 and include a comment.  

##### 仅处理特定异常情况：


In [ ]:
ToolStrategy(
    schema=ProductRating,
    handle_errors=ValueError  # Only retry on ValueError, raise others
)


如果handle_errors指定了异常类型，则代理仅在引发的异常类型为指定类型时才会重试（使用默认错误消息）。在所有其他情况下，都会引发该异常。

处理多种异常类型：

In [ ]:
ToolStrategy(
    schema=ProductRating,
    handle_errors=(ValueError, TypeError)  # Retry on ValueError and TypeError
)

如果handle_errors`exceptions` 是一个异常元组，则代理仅在引发的异常属于指定类型之一时才会重试（使用默认错误消息）。在所有其他情况下，都会引发该异常。

自定义错误处理函数：

In [ ]:
def custom_error_handler(error: Exception) -> str:
    if isinstance(error, StructuredOutputValidationError):
        return "There was an issue with the format. Try again.
    elif isinstance(error, MultipleStructuredOutputsError):
        return "Multiple structured outputs were returned. Pick the most relevant one."
    else:
        return f"Error: {str(error)}"

ToolStrategy(
    schema=ToolStrategy(Union[ContactInfo, EventDetails]),
    handle_errors=custom_error_handler
)

在StructuredOutputValidationError：

================================= Tool Message =================================.     
Name: ToolStrategy

There was an issue with the format. Try again.


在MultipleStructuredOutputsError：

================================= Tool Message ================================= 
Name: ToolStrategy 

Multiple structured outputs were returned. Pick the most relevant one.

关于其他错误：  
================================= Tool Message =================================. 
Name: ToolStrategy

Error: <error message>

无错误处理：

```python
response_format = ToolStrategy(
    schema=ProductRating,
    handle_errors=False  # All errors raised
)

```


